# **BilSTM**

BiLSTM — это двунаправленная LSTM: она смотрит на последовательность и слева-направо, и справа-налево одновременно.

По сути это “LSTM с двумя мозгами”:

первый читает ряд как обычно: от прошлого к будущему;

второй — наоборот: от будущего к прошлому;

их состояния склеиваются и подаются дальше

hₜ = [ h⃗ₜ ; h⃖ₜ ]


h⃗ₜ — LSTM вперёд по времени

h⃖ₜ — LSTM назад

[ ; ] — конкатенация

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import torch.nn as nn
import torch
import torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results
from preprocessing.target import ttp_target
from metrics.Metrics import merged_metrics
from preprocessing.ForTorch import set_seed, DEVICE, SeqDataset, torch_predict, train_torch_classifier, train_tabnet_ttp
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

...

In [ ]:
class BiLSTMCls(nn.Module):
    def init(self, n_features, hidden=64, n_classes=3, dropout=0.2):
        super().init()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden,
            batch_first=True,
            bidirectional=True
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden*2, n_classes)

    def forward(self, x):   # x: (B, L, F)
        out, _ = self.lstm(x)
        h = out[:, -1, :]          # last step
        h = self.drop(h)
        return self.fc(h)

def train_bilstm_ttp(df, train_size, test_size, step, seq_len=64):
    set_seed(42)
    splitter = prep(
        df=df,
        target_fn=ttp_target,
        target_name="ttp",
        target_col="TTP_class",
        horizons=[12,24,48],
        train_size=train_size,
        test_size=test_size,
        step=step,
        target_kwargs={"n_classes":3},
        scale_cols=[
            "Open","High","Low","Close",
            "Alligator_Jaw","Alligator_Teeth","Alligator_Lips",
            "AO","AddOn_Anchor_Level","AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:
        # seq датасеты
        tr_ds = SeqDataset(X_train, y_train, seq_len=seq_len)
        te_ds = SeqDataset(X_test,  y_test,  seq_len=seq_len)

        tr_loader = DataLoader(tr_ds, batch_size=256, shuffle=True, drop_last=False)
        te_loader = DataLoader(te_ds, batch_size=512, shuffle=False, drop_last=False)

        model = BiLSTMCls(n_features=X_train.shape[1], hidden=64, n_classes=3, dropout=0.2)

        y_pred = train_torch_classifier(
            model=model,
            train_loader=tr_loader,
            test_loader=te_loader,
            epochs=15,
            lr=1e-3
        )

        # y_test тоже надо “сдвинуть” под seq_len
        y_test_seq = y_test[seq_len-1:]
        metrics = merged_metrics(y_test_seq, y_pred)

        append_results({
            "task_type":"classification",
            "model_name":"BiLSTM",
            "model_family":"rnn",
            "model_params":{"hidden":64,"dropout":0.2,"seq_len":seq_len},
            "target_name":"ttp",
            "target_variant":"3class",
            "horizons":"12_24_48",
            **metrics
        })